# Visium MM — per-sample QC cutoff exploration

Copied from `ALL_MM_simple.ipynb`. Same loading and processing; the QC section
is replaced with tools for choosing a cutoff **per sample**, because the six
captures do not share a distribution.

## Terms used throughout

**Spot.** Visium does not measure single cells. The slide is printed with a
grid of circular capture areas, each 55 micrometres across, and each one
collects RNA from whatever tissue sits on top of it — typically a handful of
cells. One spot is one row of the data matrix.

**UMI.** Unique Molecular Identifier. A random barcode attached to each RNA
molecule before amplification. Counting UMIs counts *molecules*; counting
sequencing reads would count the same molecule many times over, because
amplification copies it. `total_counts` for a spot is its number of UMIs.

**`n_genes_by_counts`.** How many distinct genes were seen at least once in
that spot. A spot with 40 UMIs spread across 38 genes is different from one
with 40 UMIs all from 2 genes, and this column is what separates them.

**`in_tissue`.** Space Ranger compares the spot grid against the tissue
photograph and marks each spot 1 if tissue covers it, 0 if it is bare slide.
Every spot in this notebook is `in_tissue == 1` — verified, see the QC section.

## How to work through this

Read down in order. Sections 1 and 2 look at the data and at what each
candidate rule would do. Section 3 is where you choose. Sections 4 and 5 check
the choice and apply it.

In [ ]:
import warnings

warnings.filterwarnings("ignore")  # scanpy/anndata/squidpy chatter

import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq
import harmonypy as hm
import scipy.sparse as sp

## Load

In [ ]:
OUTS_BASE = "/home/workspace/PCD_data/visium/data/outs"
SAMPLES = ["TIS05393", "TIS04788", "TIS05392", "TIS05394", "TIS04789", "TIS05397"]

adatas = {}
for sample in SAMPLES:
    adata = sq.read.visium(os.path.join(OUTS_BASE, sample))
    adata.var_names_make_unique()
    adata.obs_names_make_unique()
    # Space Ranger's library id is not our short sample id; keep it for sc.pl.spatial
    adata.obs["library_id"] = list(adata.uns["spatial"])[0]
    adatas[sample] = adata

combined_adatas = sc.concat(adatas, label="dataset_name", join="outer")
combined_adatas.obs_names_make_unique()

# sc.concat drops uns; restore it so the tissue images stay available
combined_adatas.uns["spatial"] = {lib: meta for adata in adatas.values()
                                  for lib, meta in adata.uns["spatial"].items()}

print(f"{combined_adatas.n_obs} spots x {combined_adatas.n_vars} genes")
print(combined_adatas.obs["dataset_name"].value_counts())

## QC metrics

This adds three columns describing each spot. It does not remove anything and
does not change the counts — matching the original notebook, which filters
nothing.

**What gets added.** `total_counts` (UMIs per spot), `n_genes_by_counts`
(distinct genes per spot), and `pct_counts_mt`.

**`pct_counts_mt`.** The percentage of a spot's UMIs that came from
mitochondrial genes — genes encoded in the mitochondrion rather than the
nucleus, named with an `MT-` prefix. The usual reasoning: when a cell's outer
membrane ruptures, nuclear RNA leaks out while mitochondria, having their own
membrane, stay put. A high mitochondrial percentage is therefore read as a sign
the cell was damaged.

**That reasoning does not transfer cleanly here, and the numbers say so.** This
is a probe-based assay: rather than capturing all RNA, it uses a fixed panel of
probes, and that panel contains only 11 `MT-` genes out of 18,085. On TIS04788
the median mitochondrial percentage is 0.00, and 59% of spots have no
mitochondrial UMIs at all. A spot reading "50% mitochondrial" can be 1
mitochondrial UMI out of 2 total.

**Consequence.** Treat `pct_counts_mt` as close to uninformative in this
dataset. It is kept for continuity and documented where it is applied, but no
cutoff should be chosen on the strength of it.

**Why the original could skip this cell.** It read `.h5ad` files that already
carried these columns. We read Space Ranger output directly, so they have to be
computed.

In [ ]:
combined_adatas.var["mt"] = combined_adatas.var_names.str.startswith("MT-")
sc.pp.calculate_qc_metrics(combined_adatas, qc_vars=["mt"], percent_top=None,
                           log1p=False, inplace=True)

combined_adatas.obs[["n_genes_by_counts", "total_counts", "pct_counts_mt"]].describe()

## 1. Per-sample distributions

Three views of each sample, one row per sample.

**Left and middle — histograms on a log axis.** A histogram counts how many
spots fall in each range of values. The x-axis is `log10(value + 1)` rather
than the raw value: most spots have small counts and a few have very large
ones, so on a plain axis every spot piles into the leftmost bar and the shape
is invisible. Taking logs spreads the small values out. The `+ 1` exists
because `log10(0)` is undefined and some spots genuinely have zero.

**What to look for: one hump or two.** One hump means the spots form a single
population that gets gradually thinner toward the low end; any cutoff you pick
is a judgement about how much signal you require. Two humps separated by a dip
means two populations — spots that captured RNA, and spots that essentially did
not. The dip between them is a natural place to cut. Section 2 tests for this
formally rather than by eye.

**Right — the knee curve.** Sort every spot from highest UMI count to lowest,
then plot count against position in that sorted list, both axes logarithmic.
The name comes from the shape: a flat stretch on the left where well-captured
spots differ only modestly from each other, then a bend downward, then a steep
drop through the near-empty spots. The bend is the "knee". It is the same
information as the histogram, arranged so that the boundary shows up as a
change in slope rather than as a dip.

In [ ]:
fig, axes = plt.subplots(len(SAMPLES), 3, figsize=(15, 3.2 * len(SAMPLES)), squeeze=False)

for row, sample in zip(axes, SAMPLES):
    sub = combined_adatas.obs[combined_adatas.obs["dataset_name"] == sample]
    counts = sub["total_counts"].to_numpy()
    genes = sub["n_genes_by_counts"].to_numpy()

    row[0].hist(np.log10(counts + 1), bins=60, color="steelblue")
    row[0].set_xlabel("log10(UMI + 1)")
    row[0].set_title(f"{sample} — counts (median {np.median(counts):.0f})")

    row[1].hist(np.log10(genes + 1), bins=60, color="steelblue")
    row[1].set_xlabel("log10(genes + 1)")
    row[1].set_title(f"{sample} — genes (median {np.median(genes):.0f})")

    row[2].plot(np.arange(1, len(counts) + 1), np.sort(counts)[::-1], lw=1.5)
    row[2].set_xscale("log")
    row[2].set_yscale("log")
    row[2].set_xlabel("spot rank")
    row[2].set_ylabel("UMI")
    row[2].set_title(f"{sample} — knee")

fig.tight_layout()

### The same three metrics, in space

The histograms above show *how many* spots have a given value. They cannot show
*where* those spots are. This does.

Rows are samples, columns are the three QC metrics, drawn over the H&E image.

**What to look for.**

*A smooth gradient across the section* — usually technical. Reagent reaching one
edge better than the other, or the section being thinner at one end. Spots lost
to a cutoff here are a processing artifact.

*A sharp-edged patch* — check it against the H&E underneath. If the patch lines
up with a visible tissue structure, low counts there may be biology, and a
cutoff would delete a real region. If it lines up with nothing visible, it is
more likely a fold, a bubble, or a tear.

*A thin rim of low values at the tissue edge* — normal. Edge spots are partly
covered by tissue, so they capture less. Losing them is rarely a problem.

**Two things about the colour scales.**

*Each panel is scaled to itself.* A bright spot on TIS05392 and a bright spot on
TIS05393 are not the same number of molecules — the medians differ by a factor
of 40. Compare patterns within a panel; do not compare brightness across panels.

*The top 1% is clipped* (`vmax="p99"`). Counts are heavily right-skewed, so a
handful of very high spots would otherwise take the whole colour range and
flatten everything else to one shade.

**Expect the mitochondrial column to look almost blank.** As set out in the QC
metrics section, the probe panel carries only 11 `MT-` genes and the median
spot has zero mitochondrial UMIs. A near-empty panel here is the correct
result, not a plotting failure.

In [ ]:
QC_COLUMNS = ["total_counts", "n_genes_by_counts", "pct_counts_mt"]

fig, axes = plt.subplots(len(SAMPLES), len(QC_COLUMNS),
                         figsize=(5.5 * len(QC_COLUMNS), 5 * len(SAMPLES)), squeeze=False)

for i, sample in enumerate(SAMPLES):
    spots = combined_adatas[combined_adatas.obs["dataset_name"] == sample]
    library = spots.obs["library_id"].iloc[0]
    for j, column in enumerate(QC_COLUMNS):
        sc.pl.spatial(spots, color=column, library_id=library, vmax="p99",
                      ax=axes[i, j], show=False,
                      title=f"{sample} — {column}\n"
                            f"median {spots.obs[column].median():.0f}")

fig.tight_layout()

## 2. What each rule suggests

Three ways of picking a cutoff, each computed on each sample's own values.
Nothing is applied here.

**`n_cut`** is how many spots a rule removes **by its own criteria alone**,
with the shared mitochondrial cut excluded. Read it before trusting any
threshold. A rule can return a number that looks perfectly reasonable and still
remove nothing, and without this column you would not find out for three more
cells.

---

### quantile 10%

**Percentile.** Sort every value from lowest to highest; the 10th percentile is
the value standing one tenth of the way along that sorted list. Ten percent of
spots are at or below it, by construction.

**So this rule cuts exactly 10% of every sample, always.** Whether the capture
is pristine or ruined, the answer is 10%. It is a **budget, not a criterion**:
you have decided in advance how much to spend, and the threshold lands wherever
the data happens to sit at that rank.

**When that is what you want.** Equal retention across samples, so downstream
comparisons are not skewed by one sample keeping far more spots than another.

**When it is not.** Deciding whether a spot is actually bad. On TIS05392 this
rule returns `min_counts = 1` — it removes spots with 0 or 1 UMI and keeps
everything from 2 upward, in a sample whose median is 6.

---

### MAD 3

**Deviation.** For each spot, the distance between its value and the middle of
the distribution: `|value − median|`.

**MAD — median absolute deviation.** The median of all those distances. It
describes how spread out the values are. The familiar measure of spread is the
standard deviation, but that one is computed from squared distances, so a
handful of extreme values inflate it badly — and extreme values are exactly
what a QC rule is looking for. A median of distances barely moves when a few
points are extreme, so the spread estimate is not corrupted by the outliers you
are trying to catch.

**The rule.** `median − k × 1.4826 × MAD`. Anything below that is an outlier.

**Why the log first.** Counts are right-skewed: most spots are small, a few are
very large, so the distribution has a long tail on the high side and none on
the low side. MAD assumes roughly equal spread either side of the middle.
Taking `log1p` (log of value + 1) pulls the long tail in and makes the shape
closer to symmetric. `expm1` converts the resulting threshold back to counts.

**Why 1.4826.** A scaling constant that makes MAD comparable to a standard
deviation for normally distributed data, so "3 MADs" means roughly what "3
standard deviations" means. *(Standard result, not verified against a source
here.)*

**Why k = 3.** Three standard deviations from the middle covers about 99% of a
normal distribution, so beyond that is conventionally "unusual". It is the
robust analogue of the 3-sigma rule, not a biological quantity.

**What it assumes.** Two things. That most of the dataset is high quality — the
median and MAD are supposed to describe *good* spots, so the rule can measure
distance from them. And that QC values are independent of biology, which is
questionable in any tissue where some cell types simply carry less RNA.

---

### MAD 3 has failed on this data

The first assumption breaks here, and the failure is total.

**Step 1 — most spots are near zero.** On TIS05392 the median spot has 6 UMIs;
on TIS04788, 12. The median is describing near-empty spots, not good ones.

**Step 2 — the near-empty spots also inflate the spread.** They sit far from
the middle, so the typical distance is large. MAD on `log1p` counts comes out
around 0.85 to 1.49.

**Step 3 — three MADs is then enormous.** `3 × 1.4826 × MAD` is 3.8 to 6.6 on
the log scale. Undoing the log, that is a factor of roughly 45× to 750× below
the median.

**Step 4 — the threshold goes below zero.** A median of 12 counts divided by
several hundred is less than 1, and on the log scale it comes out negative:

```
          median counts  median log1p  MAD log1p  3*1.4826*MAD  med-3MAD (log)  -> counts
TIS05393            250          5.53       1.12          4.96            0.56       0.8
TIS05394            143          4.97       0.95          4.23            0.74       1.1
TIS05397             21          3.09       1.01          4.50           -1.41      -0.8
TIS04788             12          2.56       1.18          5.24           -2.68      -0.9
TIS04789             65          4.19       1.49          6.61           -2.42      -0.9
TIS05392              6          1.95       0.85          3.77           -1.82      -0.8
```

**Step 5 — a negative threshold can never fire.** Counts and gene numbers are
never negative, so `total_counts <= -0.9` is false for every spot. The
criterion is inert.

**Measured result.** MAD 3's own criteria remove **17 of 21,839 spots
(0.08%)**, all in TIS05393. It removes nothing at all on the other five —
including TIS05394, whose threshold is positive (1.1) but still sits below
every observed value.

**The consequence for every figure below.** Wherever "MAD 3" appears to remove
around 0.5% of spots, those spots were removed by the mitochondrial filter, not
by MAD. The two are easy to confuse because they arrive in the same column.

**Note the direction of the failure.** The worse a sample is, the more its
near-empty spots inflate its own MAD, the wider its interval, and the less the
rule cuts. It fails hardest exactly where it is most needed.

**Why it is kept.** Its failure is a fact about this data worth recording. It is
not repaired by lowering `k` or clipping the threshold at zero — both would
hide a violated assumption rather than address it.

---

### otsu valley

**The question it answers.** If the values form two groups, where is the
boundary between them?

**How it searches.** Bin the values into a histogram. Consider every possible
split point. For each one, the spots fall into a low group and a high group,
and you can compute each group's share of the total (`w0`, `w1`) and each
group's average value (`mu0`, `mu1`). Score the split as
`w0 × w1 × (mu0 − mu1)²` — large when the two groups have very different
averages *and* neither group is tiny. Pick the split with the highest score.

**What that score is.** The between-class variance: how much of the overall
spread is explained by the difference between the two groups rather than by
variation inside them. Maximising it is the same as minimising the spread
within the groups, because the total is fixed. *(Standard result, not verified
against a source here.)*

**Where it comes from.** Image binarisation — separating dark pixels from light
ones (Otsu 1979). dropkick applies it to the gene-count histogram to separate
uninformative barcodes from real cells.

**What it assumes.** That two groups exist. Given one group it still returns a
split, somewhere near the middle, and that split means nothing. The bimodality
check below tests this for each sample.

**A simplification here.** dropkick's version is multi-level: three classes,
with an explicit undecided band in the middle for cases that are genuinely
ambiguous. The implementation in this notebook is two-class and forces every
spot to one side.

**Its 37–70% cut rate is not obviously too aggressive.** Every spot in this
object is `in_tissue == 1`, verified against `tissue_positions.csv` and against
Space Ranger's own `Fraction of Spots Under Tissue`. So the boundary otsu finds
cannot be tissue-versus-bare-slide. It is tissue that captured well versus
tissue that did not. Space Ranger reports median UMIs of 6, 12 and 21 for
TIS05392, TIS04788 and TIS05397, with only 27–33% of reads landing in tissue
spots at all. If most spots on those sections really are near-empty, a rule
that removes most of them is working, not overreaching.

In [ ]:
def otsu(x, nbins=128):
    """Threshold splitting x into two groups with maximum between-class variance."""
    hist, edges = np.histogram(x, bins=nbins)
    centers = (edges[:-1] + edges[1:]) / 2
    w0 = np.cumsum(hist)
    w1 = hist.sum() - w0
    csum = np.cumsum(hist * centers)
    mu0 = csum / np.where(w0 == 0, 1, w0)
    mu1 = (csum[-1] - csum) / np.where(w1 == 0, 1, w1)
    return centers[np.argmax(w0 * w1 * (mu0 - mu1) ** 2)]


def mad_floor(x, k=3):
    """median - k * MAD, on the log1p scale, returned on the original scale.

    Returned as-is when non-positive. Clipping it to 0 would hide that the
    rule's assumption failed on this sample; the check below reports it.
    """
    logged = np.log1p(x)
    med = np.median(logged)
    mad = np.median(np.abs(logged - med))
    return float(np.expm1(med - k * 1.4826 * mad))


RULES = {
    "quantile 10%": lambda x: float(np.quantile(x, 0.10)),
    "MAD 3": mad_floor,
    "otsu valley": lambda x: float(10 ** otsu(np.log10(x + 1)) - 1),
}

rows = {}
for sample in SAMPLES:
    sub = combined_adatas.obs[combined_adatas.obs["dataset_name"] == sample]
    row = {}
    for name, rule in RULES.items():
        row[f"counts {name}"] = round(rule(sub["total_counts"].to_numpy()), 1)
        row[f"genes {name}"] = round(rule(sub["n_genes_by_counts"].to_numpy()), 1)
    rows[sample] = row
suggested = pd.DataFrame(rows).T

# How many spots each rule's OWN criteria remove. Mito is excluded on purpose:
# it is a shared constant, and folding it in here would let a rule that cuts
# nothing still show a non-zero number (which is exactly what hid the MAD
# failure previously).
for name in RULES:
    counts = []
    for sample in SAMPLES:
        sub = combined_adatas.obs[combined_adatas.obs["dataset_name"] == sample]
        fired = ((sub["n_genes_by_counts"] <= suggested.loc[sample, f"genes {name}"])
                 | (sub["total_counts"] <= suggested.loc[sample, f"counts {name}"]))
        counts.append(int(fired.sum()))
    suggested[f"n_cut {name}"] = pd.Series(counts, index=SAMPLES)

suggested = suggested[[f"{p} {name}" for name in RULES
                       for p in ("counts", "genes", "n_cut")]]

# A threshold at or below 0 can never fire: counts and genes are non-negative.
# Surface it by name rather than letting a silent 0 propagate three cells on.
for sample in suggested.index:
    for name in RULES:
        for metric in ("counts", "genes"):
            value = suggested.loc[sample, f"{metric} {name}"]
            if value <= 0:
                print(f"  !! {name:12s} {metric:6s} {sample}: threshold {value:.2f} "
                      f"<= 0 — cannot fire, this criterion is inert")
    for name in RULES:
        if suggested.loc[sample, f"n_cut {name}"] == 0:
            print(f"  !! {name:12s}        {sample}: cuts 0 spots")

print("\nSuggested cutoffs per sample (n_cut = spots removed by that rule alone,")
print("excluding the shared mitochondrial cut)")
print(suggested.to_string())

### Does a valley exist? (bimodality check)

**Why this cell exists.** The `otsu valley` rule assumes the values form two
groups. On a single-group distribution it still returns a number — somewhere
near the middle — and that number means nothing. Trusting it without checking
is the failure mode. This tests the assumption per sample.

**Gaussian mixture model.** A model that describes the data as coming from some
number of overlapping bell curves. Fit it with one bell curve and it says
"these are one population". Fit it with two and it says "these are two
populations with these centres and spreads". Both fits are run on
`log1p(value)`, the same log scale as the histograms above.

**BIC.** Bayesian Information Criterion. A single number scoring how well a
model fits, minus a penalty for how many parameters the model uses. Lower is
better. The penalty matters: a two-curve model has more freedom and will always
fit the data at least slightly better than a one-curve model, so raw fit
quality would always pick two. BIC charges for that extra freedom, so two
curves only win when they explain enough extra structure to pay for
themselves.

**How to read the output.** If the two-curve BIC is lower, the sample has two
genuine groups and its otsu threshold is meaningful. If the one-curve BIC is
lower, it does not, and the row is marked `UNIMODAL — otsu unreliable`.

**Watch the margin, not just the verdict.** A win of 10 BIC points and a win of
3,000 are both "bimodal", but they are not equally convincing.

In [ ]:
from sklearn.mixture import GaussianMixture

rows = {}
for sample in SAMPLES:
    sub = combined_adatas.obs[combined_adatas.obs["dataset_name"] == sample]
    row = {}
    for metric in ("total_counts", "n_genes_by_counts"):
        x = np.log1p(sub[metric].to_numpy()).reshape(-1, 1)
        bic1 = GaussianMixture(1, random_state=0).fit(x).bic(x)
        bic2 = GaussianMixture(2, random_state=0).fit(x).bic(x)
        row[f"{metric} BIC 1c"] = round(bic1)
        row[f"{metric} BIC 2c"] = round(bic2)
        row[f"{metric} verdict"] = "bimodal" if bic2 < bic1 else "UNIMODAL — otsu unreliable"
    rows[sample] = row

bimodality = pd.DataFrame(rows).T
# margin = how many BIC points the 2-curve model wins by, on counts. A big
# margin means two genuine populations; a margin near 0 is a coin flip.
bimodality["counts margin"] = (bimodality["total_counts BIC 1c"]
                               - bimodality["total_counts BIC 2c"]).astype(int)

print("1- vs 2-component GMM on log1p values (lower BIC wins)")
print(bimodality.to_string())

### What each rule cuts, in space

Every rule applied at its own per-sample numbers. Rows are tissues, columns are
rules. Nothing is filtered — this is the comparison that decides which rule you
trust.

**What the colours mean.** Each spot is labelled with why it would be removed:
`low genes`, `low counts`, `high % mt`, or `multiple` when more than one
applies. `kept` is grey. Colouring by reason rather than by a single cut/keep
flag shows which criterion is doing the work — and on these samples it is
almost entirely counts and genes, not mitochondrial percentage.

**What to look for.** Removed spots along the tissue edge suggest a thin rim
where the section was thinner and captured less — usually fine to lose. Removed
spots forming a solid block in the middle of tissue mean the rule is deleting a
region, which is the thing to be careful about.

The mitochondrial cutoff is held at `MAX_MT` for all three rules, because none
of the three rules produces a mitochondrial threshold.

In [ ]:
# Mitochondrial cutoff, held constant across samples while genes/counts are
# derived per sample. Measured, not assumed: these are human bone marrow
# biopsies run on the Visium Human Transcriptome Probe Set v2.0, which carries
# 11 MT- genes. On TIS04788 the median MT UMIs per spot is 0, 59% of spots have
# zero MT counts, and median pct_counts_mt is 0.00 — so a spot reading "50%
# mitochondrial" can be 1 MT UMI out of 2. The metric is dominated by depth,
# not by damage, and only ~0.6% of spots exceed 20%. Deriving it per sample
# would fit noise, so it stays a documented constant and is not load-bearing.
MAX_MT = 20

REASONS = ["kept", "low genes", "low counts", "high % mt", "multiple"]
PALETTE = {"kept": "#d9d9d9", "low genes": "#1f77b4", "low counts": "#ff7f0e",
           "high % mt": "#d62728", "multiple": "#9467bd"}


def cut_reason(obs, cutoffs):
    """Per-spot label: kept, the single reason it failed, or multiple.

    obs is passed in rather than closed over, so the caller decides which
    frame is being labelled and when.
    """
    fail = pd.DataFrame(False, index=obs.index, columns=REASONS[1:4])
    for sample, t in cutoffs.items():
        m = (obs["dataset_name"] == sample).to_numpy()
        fail.loc[m, "low genes"] = obs.loc[m, "n_genes_by_counts"] <= t["min_genes"]
        fail.loc[m, "low counts"] = obs.loc[m, "total_counts"] <= t["min_counts"]
        # 383 spots have zero total counts, so pct_counts_mt is NaN (0/0).
        # NaN >= 20 is False, meaning they would pass the mito test silently.
        # fillna(0) makes that explicit and keeps the behaviour: a spot with no
        # counts has no mitochondrial reads either, and the count criterion
        # already removes it.
        mito = obs.loc[m, "pct_counts_mt"].fillna(0)
        fail.loc[m, "high % mt"] = mito >= t["max_mt"]
    reason = fail.idxmax(axis=1).where(fail.any(axis=1), "kept")
    reason[fail.sum(axis=1) > 1] = "multiple"
    return pd.Categorical(reason, categories=REASONS)


for name in RULES:
    combined_adatas.obs[name] = cut_reason(combined_adatas.obs, {
        sample: {"min_genes": suggested.loc[sample, f"genes {name}"],
                 "min_counts": suggested.loc[sample, f"counts {name}"],
                 "max_mt": MAX_MT}
        for sample in SAMPLES})

# read after the writes above, explicitly, not through an alias bound earlier
print("% of spots cut")
print(combined_adatas.obs.groupby("dataset_name", observed=True)[list(RULES)]
      .agg(lambda s: 100 * (s != "kept").mean()).round(1).to_string())

fig, axes = plt.subplots(len(SAMPLES), len(RULES),
                         figsize=(4.5 * len(RULES), 4.5 * len(SAMPLES)), squeeze=False)

for i, sample in enumerate(SAMPLES):
    spots = combined_adatas[combined_adatas.obs["dataset_name"] == sample]
    for j, name in enumerate(RULES):
        n_cut = int((spots.obs[name] != "kept").sum())
        sc.pl.spatial(spots, color=name, library_id=spots.obs["library_id"].iloc[0],
                      palette=PALETTE, ax=axes[i, j], show=False,
                      legend_loc="right margin" if (i, j) == (0, len(RULES) - 1) else None,
                      title=f"{sample} — {name}\n"
                            f"genes>{suggested.loc[sample, f'genes {name}']:.0f}, "
                            f"counts>{suggested.loc[sample, f'counts {name}']:.0f}\n"
                            f"{n_cut} cut ({100 * n_cut / spots.n_obs:.0f}%)")

fig.tight_layout()

### Are the discards spatially structured?

**The worry this cell addresses.** A cutoff decides which spots to throw away
using only each spot's own numbers — it never looks at where the spot sits on
the tissue. If the low-count spots happen to be one contiguous region, the
cutoff quietly deletes that region. If that region is a real anatomical
structure rather than a damaged patch, the analysis loses it and nothing in the
numbers announces the loss.

**Spatial autocorrelation.** The tendency of nearby things to resemble each
other more than distant things. Height above sea level has strong spatial
autocorrelation: two points a metre apart are almost the same altitude. A coin
flip at each point has none.

**Moran's I.** A number summarising spatial autocorrelation for one variable
across one set of locations. Compare each spot's value against the average of
its immediate neighbours, and average that agreement over all spots. It runs
from about −1 to +1:

- **near +1** — neighbours match. Discarded spots sit next to other discarded
  spots, so the discards form solid patches.
- **near 0** — no relationship. Discards are scattered.
- **negative** — neighbours differ more than chance, like a checkerboard.

With no spatial pattern the expected value is `−1/(n−1)`, which for thousands
of spots is a hair below zero rather than exactly zero. *(Standard result, not
verified against a source here.)* The `shuffled` column demonstrates this: it
is the same discard labels randomly reassigned to spots, so it must be
unstructured, and it comes out at about ±0.02. That is the yardstick — anything
near it is scattered, anything far above it is patchy.

**Two reasons this table does not rank the rules.**

*First, the score depends on how much you cut.* The rules discard wildly
different amounts: otsu 37–70% of spots, quantile about 10%, MAD 3 essentially
0% by its own criteria, SpotSweeper 0.3–2.3%. A rule that discards 20 spots
cannot make a solid patch out of them, so its low score is arithmetic rather
than evidence about the rule. MAD 3's near-zero value here is exactly that.

*Second, three of the four rules are the same rule underneath.* `quantile 10%`,
`MAD 3` and `otsu valley` all sort spots by the same thing — `total_counts`, or
`n_genes_by_counts`, low to high — and differ only in where along that sorted
list they draw the line. Draw the line at the same position and all three
produce an identical set of discarded spots. Their score differences therefore
measure where they cut, not how they decide.

**Consequence.** Use this table to ask "does this rule remove a region?", one
rule at a time. Do not use it to pick a winner. The matched-rate sweep further
down does that comparison properly.

**Why `n_neighs=6` here and `k=36` in SpotSweeper.** Visium spots sit on a
hexagonal grid, so each spot has 6 immediate neighbours; go out a second ring
and you have 18, a third ring and you have 36. Moran's I uses 6 because it is
asking whether a spot resembles the spots touching it, and averaging over a
wide area would blur that. SpotSweeper uses 36 because it needs a large enough
group to compute a stable median and spread. Different jobs, different sizes.

In [ ]:
rng = np.random.default_rng(0)
rows = []

for sample in SAMPLES:
    spots = combined_adatas[combined_adatas.obs["dataset_name"] == sample].copy()
    sq.gr.spatial_neighbors(spots, coord_type="grid", n_neighs=6)

    columns = []
    for name in RULES:
        spots.obs[f"{name} cut"] = (spots.obs[name] != "kept").astype(float)
        columns.append(f"{name} cut")
    spots.obs["shuffled"] = rng.permutation(spots.obs[columns[0]].to_numpy())
    columns.append("shuffled")

    result = sq.gr.spatial_autocorr(spots, attr="obs", genes=columns, mode="moran",
                                    n_perms=99, seed=0, copy=True,
                                    show_progress_bar=False)
    rows.append(result["I"].rename(sample))

moran = pd.DataFrame(rows)
print("Moran's I of the cut/kept label  (0 = scattered, 1 = one solid patch)")
print(moran.round(3).to_string())

### A spatially-aware rule: SpotSweeper local outliers

**The idea.** Every rule so far compares a spot against its whole sample: is
this spot low compared to the other 5,640 spots on this slide? That question
has a built-in flaw — if a large area captured badly, all its spots are low,
and a sample-wide comparison flags the whole area. SpotSweeper changes the
comparison group. It asks: is this spot low compared to *the spots immediately
around it*?

**Consequence of that change.** A uniformly low-signal region is no longer
flagged, because each spot in it looks normal for its own surroundings. A
single bad spot sitting in a good region is flagged, because its neighbours are
fine and it is not.

**Robust z-score.** A z-score says how far a value sits from the middle,
measured in units of spread: a z of −3 means "three spreads below the middle".
The ordinary version uses the mean and standard deviation, both of which are
dragged around by the very extreme values you are hunting for. The robust
version swaps in the median and the median absolute deviation, neither of which
moves much when a few points are extreme. SpotSweeper computes this for each
spot against its 36 nearest neighbours and flags spots past a cutoff.

**k = 36 is the third ring.** Three concentric rings of neighbours on the
hexagonal grid. Square-grid platforms use 48.

**API, read from the installed package** (`spotsweeper==1.0.0`,
github.com/danielchen05/spotsweeper_py) rather than written from memory:

```
local_outliers(adata, metric='detected', direction='lower', n_neighbors=36,
               sample_key='sample_id', log=True, cutoff=3.0, workers=1,
               coord_key='spatial')
```

It writes two columns per metric into `obs`: `{metric}_z`, the continuous
score, and `{metric}_outliers`, the boolean flag. Both are carried back below —
the flags for the rule, the scores for the matched-rate sweep.

**The limitation, and it matters here.** *(This is reasoning, not a sourced
claim.)* The neighbourhood is the yardstick, so a damaged area larger than the
neighbourhood becomes its own normal. Every spot in it is compared against
other bad spots, finds itself typical, and passes. The method is blind to
failures bigger than about 36 spots.

That is not hypothetical on this data. TIS05392 has a median of 6 UMIs across
5,640 spots — the bad area is most of the section, far larger than any
neighbourhood. The measured cut rates below (0.3–2.3%) are consistent with the
method finding almost nothing there.

**Consequence.** Keep a whole-sample rule alongside this one. SpotSweeper
answers "which spots are anomalous for their surroundings", which is a
different question from "which spots have too little signal to use", and this
dataset needs both answered.

**Unverified.** Whether a cutoff of 3.0 is appropriate for bone marrow. Read
the maps.

In [ ]:
# the package __init__ does not re-export; the function lives in the submodule
from spotsweeper.local_outliers import local_outliers

SS_CUTOFF = 3.0
SS_NEIGHBORS = 36

ss = combined_adatas.copy()
for metric, direction in [("n_genes_by_counts", "lower"), ("total_counts", "lower")]:
    local_outliers(ss, metric=metric, direction=direction, n_neighbors=SS_NEIGHBORS,
                   sample_key="dataset_name", cutoff=SS_CUTOFF, coord_key="spatial")

# carry BOTH outputs back: the booleans for the rule, and the continuous
# robust z-scores, which the matched-rate sweep below needs
for metric in ("n_genes_by_counts", "total_counts"):
    combined_adatas.obs[f"{metric}_z"] = ss.obs[f"{metric}_z"].to_numpy()

low_genes = ss.obs["n_genes_by_counts_outliers"].to_numpy().astype(bool)
low_counts = ss.obs["total_counts_outliers"].to_numpy().astype(bool)

fail = pd.DataFrame({"low genes": low_genes, "low counts": low_counts,
                     "high % mt": np.zeros(len(low_genes), dtype=bool)},
                    index=combined_adatas.obs.index)
reason = fail.idxmax(axis=1).where(fail.any(axis=1), "kept")
reason[fail.sum(axis=1) > 1] = "multiple"
combined_adatas.obs["spotsweeper"] = pd.Categorical(reason, categories=REASONS)

print(f"SpotSweeper local outliers (k={SS_NEIGHBORS}, robust-z cutoff {SS_CUTOFF})")
print(combined_adatas.obs.groupby("dataset_name", observed=True)["spotsweeper"]
      .agg(lambda s: f"{(s != 'kept').sum()} cut ({100 * (s != 'kept').mean():.1f}%)")
      .to_string())

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for ax, sample in zip(axes.flatten(), SAMPLES):
    spots = combined_adatas[combined_adatas.obs["dataset_name"] == sample]
    n_cut = int((spots.obs["spotsweeper"] != "kept").sum())
    sc.pl.spatial(spots, color="spotsweeper", library_id=spots.obs["library_id"].iloc[0],
                  palette=PALETTE, ax=ax, show=False,
                  title=f"{sample} — spotsweeper\n{n_cut} cut "
                        f"({100 * n_cut / spots.n_obs:.1f}%)")
fig.tight_layout()

#### Is it less spatially confounded?

The same Moran's I calculation as above, now with SpotSweeper included, so the
local rule can be read against the three global ones.

**Both caveats from the previous table still apply**, and they apply to this
one too: the score depends on how many spots each rule discards, and the three
global rules share a single ordering. SpotSweeper discards 0.3–2.3% of spots,
which is little enough that a low score here is partly arithmetic.

**So treat this table as a description, not a verdict.** It tells you the
shape of each rule's discards at its own natural cut rate. It does not tell
you which rule is better, because the rules are not being asked to remove the
same amount.

The next cell removes that problem by forcing a matched number of discards.

In [ ]:
rows = []
for sample in SAMPLES:
    spots = combined_adatas[combined_adatas.obs["dataset_name"] == sample].copy()
    sq.gr.spatial_neighbors(spots, coord_type="grid", n_neighs=6)

    columns = []
    for name in list(RULES) + ["spotsweeper"]:
        spots.obs[f"{name} cut"] = (spots.obs[name] != "kept").astype(float)
        columns.append(f"{name} cut")

    result = sq.gr.spatial_autocorr(spots, attr="obs", genes=columns, mode="moran",
                                    n_perms=99, seed=0, copy=True,
                                    show_progress_bar=False)
    rows.append(result["I"].rename(sample))

print("Moran's I of the cut/kept label, global rules vs local")
print(pd.DataFrame(rows).round(3).to_string())

### Which rule for which sample?

Everything above examines one rule at a time. This puts the evidence for all
six samples in one place, so the choice in section 3 is made from a table
rather than from memory.

**The question each sample poses.** Every capture here has a population of
near-empty spots. The only thing that matters is whether that population is
tissue that failed to capture — in which case removing it is correct — or
tissue that is genuinely low in RNA, in which case removing it destroys real
biology. The columns below are three independent readings on that question.

**`median UMI`** — the middle spot's molecule count, from Space Ranger. 250 is
a healthy capture; 6 is not.

**`reads in tissue`** — the fraction of all sequencing reads that landed on a
spot covered by tissue, from Space Ranger. When this is 0.96, the chemistry
worked and the RNA went where it should. When it is 0.27, most of the
sequencing went to bare slide, which is a section-level capture failure and
strong evidence the low-signal spots are technical rather than biological.

**`bimodality margin`** — how many BIC points the two-population model beats
the one-population model by, from the check above. A large margin means the
near-empty spots really are a separate population with a boundary to find. A
margin near zero means there is one population and otsu is drawing a line
through the middle of it, which is meaningless.

**`suggested`** — a reading of those three columns, with the reason stated. It
is a starting point, not an answer.

**This table cannot close the question on its own.** The final check is section
4: look at what the rule removes on the H&E image. If the removed region is
pale and sparse, it failed and should go. If it is dense cellular marrow, the
rule is deleting biology no matter what these numbers say.

In [ ]:
# Judgement calls, not derived quantities — stated here so they can be argued
# with rather than buried in the logic below.
MARGIN_STRONG = 100    # BIC points; below this the 2-population split is a coin flip
READS_FAILED = 0.5     # fraction of reads in tissue; below this the capture failed

rows = {}
for sample in SAMPLES:
    summary = pd.read_csv(os.path.join(OUTS_BASE, sample, "metrics_summary.csv")).iloc[0]
    margin = int(bimodality.loc[sample, "counts margin"])
    reads = float(summary["Fraction Reads in Spots Under Tissue"])
    n_spots = int((combined_adatas.obs["dataset_name"] == sample).sum())

    if margin < MARGIN_STRONG:
        suggested_rule = "manual"
        why = "one population — otsu's split is a coin flip"
    elif reads < READS_FAILED:
        suggested_rule = "otsu valley"
        why = "capture failed and two populations are clear"
    else:
        suggested_rule = "check the map"
        why = "two populations but the capture worked — is the low mode real?"

    rows[sample] = {
        "median UMI": int(summary["Median UMI Counts per Spot"]),
        "reads in tissue": round(reads, 3),
        "bimodality margin": margin,
        "otsu cuts": int(suggested.loc[sample, "n_cut otsu valley"]),
        "otsu %": round(100 * suggested.loc[sample, "n_cut otsu valley"] / n_spots, 1),
        "quantile cuts": int(suggested.loc[sample, "n_cut quantile 10%"]),
        "suggested": suggested_rule,
        "why": why,
    }

decision = pd.DataFrame(rows).T
print(decision.to_string())

## 3. Your cutoffs

**How the choice is expressed.** `RULE_PER_SAMPLE` maps each sample to the name
of the rule it should use — any name from `RULES`, or `"manual"` to use the
numbers you typed into `MANUAL`. Different samples can use different rules;
given how differently the six captures behaved, that is the expected outcome
rather than a compromise.

**Why per-sample rules are defensible here.** The samples are not comparable in
quality. TIS05393 has a median of 250 UMIs per spot; TIS05392 has 6. A rule
that suits one is wrong for the other.

**What it costs you.** Once samples are filtered by different rules, the
fraction of spots retained is no longer comparable between them, and neither is
anything downstream that depends on how many spots survived. Record which rule
each sample used — the `rule` column in the report exists for that reason, so
the choice travels with the numbers.

Re-run this cell after editing to see what your choice removes.

In [ ]:
MANUAL = {
    "TIS05393": {"min_genes": 20, "min_counts": 40, "max_mt": 20},
    "TIS04788": {"min_genes": 20, "min_counts": 40, "max_mt": 20},
    "TIS05392": {"min_genes": 20, "min_counts": 40, "max_mt": 20},
    "TIS05394": {"min_genes": 20, "min_counts": 40, "max_mt": 20},
    "TIS04789": {"min_genes": 20, "min_counts": 40, "max_mt": 20},
    "TIS05397": {"min_genes": 20, "min_counts": 40, "max_mt": 20},
}

RULE_PER_SAMPLE = {
    "TIS05393": "quantile 10%",
    "TIS04788": "quantile 10%",
    "TIS05392": "quantile 10%",
    "TIS05394": "quantile 10%",
    "TIS04789": "quantile 10%",
    "TIS05397": "quantile 10%",
}

# every rule's numbers, plus MANUAL, as one lookup: CANDIDATES[rule][sample]
CANDIDATES = {name: {sample: {"min_genes": suggested.loc[sample, f"genes {name}"],
                              "min_counts": suggested.loc[sample, f"counts {name}"],
                              "max_mt": MAX_MT}
                     for sample in SAMPLES}
              for name in RULES}
CANDIDATES["manual"] = MANUAL

CUTOFFS = {sample: CANDIDATES[rule][sample] for sample, rule in RULE_PER_SAMPLE.items()}
combined_adatas.obs["cut_reason"] = cut_reason(combined_adatas.obs, CUTOFFS)

report = pd.DataFrame(CUTOFFS).T
report.insert(0, "rule", pd.Series(RULE_PER_SAMPLE))
report = report.join(pd.crosstab(combined_adatas.obs["dataset_name"], combined_adatas.obs["cut_reason"]))
report["spots"] = pd.crosstab(combined_adatas.obs["dataset_name"], combined_adatas.obs["cut_reason"]).sum(axis=1)
report["cut"] = report["spots"] - report["kept"]
report["% cut"] = (100 * report["cut"] / report["spots"]).round(1)
print(report.to_string())
print(f"\ntotal {report['cut'].sum()} of {report['spots'].sum()} spots cut")

## 4. Where those spots are

The report above says how many spots go. It cannot say *which*. A rule removing
15% of a sample is fine if those spots are scattered thin patches and serious
if they are one solid region of tissue.

Each panel shows one sample with its own cutoff in the title, spots coloured by
the reason they would be removed.

**Reading the `multiple` column in the report above.** `multiple` dominates the
discards — 787 of 851 for TIS05392 at `quantile 10%`. That is not a subtlety of
how the labelling works. With cutoffs of 1, a spot that fails both the gene
criterion and the count criterion has essentially nothing in it. Here
`multiple` means "empty", not "borderline on two separate measures".

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

for ax, sample in zip(axes.flatten(), SAMPLES):
    spots = combined_adatas[combined_adatas.obs["dataset_name"] == sample]
    n_cut = int((spots.obs["cut_reason"] != "kept").sum())
    t = CUTOFFS[sample]
    sc.pl.spatial(spots, color="cut_reason", library_id=spots.obs["library_id"].iloc[0],
                  palette=PALETTE, ax=ax, show=False,
                  title=f"{sample} — genes>{t['min_genes']}, counts>{t['min_counts']}\n"
                        f"{n_cut} cut ({100 * n_cut / spots.n_obs:.0f}%)")

fig.tight_layout()

### Appendix — method validation: do the two orderings actually differ?

**This is not a per-sample decision cell.** It answers one question about the
methods themselves, once, and it does not tell you which rule to use on any
sample. Skip it unless you want to know whether SpotSweeper earns its place.

**The question.** Earlier, SpotSweeper's discards scored much lower on Moran's
I than the global rules'. Two very different things could explain that:

1. Its ordering genuinely picks scattered spots rather than regions.
2. It only discards 0.3-2.3% of spots, and you cannot build a solid patch out
   of 1% of a slide.

Both predict a low score, so the earlier table cannot separate them.

**The test.** Force every method to discard *the same number* of spots, then
compare. That removes explanation 2. Only two orderings are genuinely
different:

- **global** — sort spots by their own raw value, lowest first. All three
  global rules share this ordering and differ only in where they stop.
- **local** — sort spots by SpotSweeper's neighbourhood z-score, lowest first.

Take the lowest 1%, 2%, 5%, 10% and 20% by each ordering and measure Moran's I
of each set.

**Result.** At a matched 10%, global scores 0.425 and local scores 0.168, and
the gap widens as the percentage rises. Explanation 1 holds: the local ordering
really does pick scattered spots where the global ordering picks regions.

**What this does not tell you.** Whether removing a region is *wrong*. A high
score means the discards form a contiguous blob; it is silent on whether that
blob is failed tissue that should go or real biology that should stay. Both
look identical to this statistic. Use the decision table and the H&E maps for
that — this cell only establishes that the two orderings are not the same tool.

---

#### A confound in this sweep, and which way it pushes

**Discrete counts.** A UMI count is a whole number. A spot has 0 UMIs, or 1, or
47. It cannot have 2.7. Compare a temperature reading, which can be 21.4837 —
that is continuous.

**Ties.** A tie means two spots have the same value, so you cannot rank one
above the other. At the top of the distribution ties are rare: there are
thousands of possible values and the spots spread thin across them. At the
bottom there are only a handful of possible values — 0, 1, 2, 3 — and thousands
of spots crowd into them. On TIS05392, 290 spots have exactly 0 counts.

**Why that breaks the sweep.** The sweep asks for the 1% lowest-count spots.
For TIS05392 that is 56 spots. But 290 spots are tied at zero, so "the lowest
56" has no unique answer.

**What the code does instead.** `np.argsort(kind="stable")` breaks the tie by
falling back to array order, which is barcode order — unrelated to where a spot
sits on the tissue. So the 56 it picks are scattered across the slide rather
than concentrated in the worst patch.

**Consequence.** A scattered set scores low on Moran's I. The global line is
therefore pushed down at small percentages. Since the claim under test is
"local is *less* clumped than global", this artifact makes the two look more
similar than they really are — it works against the claim rather than
manufacturing it, so the true gap is wider than plotted.

The local ordering does not have this problem: z-scores are continuous, so ties
are rare.

**So read the comparison from 5% rightward**, where the number of spots
strictly below the boundary exceeds the tie group on every sample.

In [ ]:
FRACTIONS = [0.01, 0.02, 0.05, 0.10, 0.20]
METRICS = ["total_counts", "n_genes_by_counts"]

rng = np.random.default_rng(0)
records, shuffled_ref = [], {}

for sample in SAMPLES:
    spots = combined_adatas[combined_adatas.obs["dataset_name"] == sample].copy()
    sq.gr.spatial_neighbors(spots, coord_type="grid", n_neighs=6)
    n_spots = spots.n_obs

    columns = []
    for metric in METRICS:
        orderings = {"global": spots.obs[metric].to_numpy(),           # raw value
                     "local": spots.obs[f"{metric}_z"].to_numpy()}     # neighbourhood z
        for fraction in FRACTIONS:
            n = max(int(round(fraction * n_spots)), 1)
            for kind, values in orderings.items():
                mask = np.zeros(n_spots)
                mask[np.argsort(values, kind="stable")[:n]] = 1.0
                column = f"{metric}|{kind}|{fraction}"
                spots.obs[column] = mask
                columns.append(column)

    # reference: a 10% mask placed at random, i.e. no spatial structure
    reference = np.zeros(n_spots)
    reference[: max(int(round(0.10 * n_spots)), 1)] = 1.0
    spots.obs["shuffled"] = rng.permutation(reference)
    columns.append("shuffled")

    result = sq.gr.spatial_autocorr(spots, attr="obs", genes=columns, mode="moran",
                                    n_perms=None, copy=True, show_progress_bar=False)
    shuffled_ref[sample] = result.loc["shuffled", "I"]
    for column in columns[:-1]:
        metric, kind, fraction = column.split("|")
        records.append({"sample": sample, "metric": metric, "kind": kind,
                        "fraction": float(fraction), "I": result.loc[column, "I"]})

sweep = pd.DataFrame(records)

for metric in METRICS:
    fig, axes = plt.subplots(2, 3, figsize=(16, 9), sharey=True)
    for ax, sample in zip(axes.flatten(), SAMPLES):
        block = sweep[(sweep["sample"] == sample) & (sweep["metric"] == metric)]
        for kind, style in [("global", "o-"), ("local", "s--")]:
            series = block[block["kind"] == kind].sort_values("fraction")
            ax.plot(100 * series["fraction"], series["I"], style, label=kind)
        ax.axhline(shuffled_ref[sample], color="grey", ls=":", label="shuffled")
        ax.set_title(sample)
        ax.set_xlabel("% of spots discarded")
        ax.set_ylabel("Moran's I")
    axes.flatten()[0].legend(fontsize=9)
    fig.suptitle(f"Matched-rate discard structure — {metric}\n"
                 "global = lowest raw value; local = lowest SpotSweeper neighbourhood z",
                 y=1.0)
    fig.tight_layout()
    plt.show()

print("Moran's I at matched discard rates (mean over samples)")
print(sweep.groupby(["metric", "kind", "fraction"])["I"].mean().round(3).to_string())

## 5. Apply

Removes the spots the preview marked, using the mask computed in section 3 —
not a second copy of the comparison logic. The preview and the filter cannot
disagree, because there is only one place the decision is made.

Everything below this point is the standard processing pipeline, unchanged from
`ALL_MM_simple.ipynb`.

In [ ]:
combined_adatas = combined_adatas[combined_adatas.obs["cut_reason"] == "kept"].copy()

print(f"{combined_adatas.n_obs} spots kept")
combined_adatas.obs["dataset_name"].value_counts()

## Process

Normalize, then reduce to principal components, then build the neighbour graph
the clustering and UMAP run on.

**Inherited from the original notebook: `sc.pp.log1p` is commented out.**

**What that means.** Normalizing rescales each spot so that spots sequenced to
different depths are comparable. The log step, which normally follows,
compresses large values relative to small ones. Without it, PCA sees raw
normalized counts, so a spot with ten times the library size exerts ten times
the pull on the components. With the log step it would exert about `log(10)`
times the pull.

**Why it matters on this data specifically.** The samples differ in depth by a
factor of 40 — median 250 UMIs on TIS05393 against 6 on TIS05392. Skipping the
log lets that depth difference dominate the components, which is exactly the
axis Harmony is then asked to correct for.

**Left as-is deliberately.** Changing it changes every result below. It is
flagged rather than fixed because it is a real choice, not an oversight to
silently repair.

In [ ]:
combined_adatas.layers["counts"] = combined_adatas.X.copy()
sc.pp.normalize_total(combined_adatas, inplace=True)
# sc.pp.log1p(combined_adatas)  # left off, as in the original

# normalize_total can densify X
combined_adatas.X = sp.csr_matrix(combined_adatas.X)

sc.pp.pca(combined_adatas, n_comps=30, use_highly_variable=False)
sc.pp.neighbors(combined_adatas, n_neighbors=15, n_pcs=30)
sc.tl.umap(combined_adatas)
sc.tl.leiden(combined_adatas)

In [ ]:
sc.pl.umap(combined_adatas,
           color=["total_counts", "n_genes_by_counts", "leiden", "dataset_name"],
           wspace=0.4)

## Harmony batch correction

In [ ]:
# harmonypy called directly (the scanpy wrapper is a version mismatch)
ho = hm.run_harmony(combined_adatas.obsm["X_pca"].astype("float64"),
                    combined_adatas.obs, "dataset_name")

# Z_corr orientation varies by harmonypy version
z = ho.Z_corr
combined_adatas.obsm["X_pca_harmony"] = z.T if z.shape[0] != combined_adatas.n_obs else z

sc.pp.neighbors(combined_adatas, use_rep="X_pca_harmony")
sc.tl.umap(combined_adatas)
sc.tl.leiden(combined_adatas, flavor="igraph", n_iterations=2, directed=False,
             resolution=0.6)

combined_adatas.obs["leiden"].value_counts().sort_index()

In [ ]:
sc.pl.umap(combined_adatas,
           color=["total_counts", "n_genes_by_counts", "leiden", "dataset_name"],
           wspace=0.15,
           ncols=2
          )

## Marker genes

In [ ]:
sc.tl.rank_genes_groups(combined_adatas, "leiden", method="wilcoxon")
sc.pl.rank_genes_groups(combined_adatas, n_genes=10, sharey=False)

## Cell types

Cluster ids are only meaningful for the run that produced them — check these
against the marker genes above before trusting them.

In [ ]:
# cluster_to_cell_type = {
#     "0": "Erythroblast_1",
#     "1": "Macrophage",
#     "2": "Monocyte",
#     "3": "Endothelial_1",
#     "4": "Monocyte/Macrophage",
#     "5": "Stromal/Fibroblast",
#     "6": "Tumor-Associated Macrophage",
#     "7": "Neoplasm_1",
#     "8": "Erythroblast_2",
#     "9": "Neutrophil",
#     "10": "Neoplasm_2",
#     "11": "Neoplasm_3",
#     "12": "Cytotoxic T Cell",
#     "13": "Neoplasm_4",
#     "14": "Myeloid",
#     "15": "Neoplasm_5",
#     "16": "Endothelial_2",
# }

# combined_adatas.obs["cell_type"] = (
#     combined_adatas.obs["leiden"].map(cluster_to_cell_type).astype("category"))

# sc.pl.umap(combined_adatas, color="cell_type", legend_fontsize=8)

## Spatial

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

for ax, sample in zip(axes.flatten(), SAMPLES):
    spots = combined_adatas[combined_adatas.obs["dataset_name"] == sample]
    sc.pl.spatial(spots, color="leiden", library_id=spots.obs["library_id"].iloc[0],
                  title=sample, ax=ax, show=False)

fig.tight_layout()

In [ ]:
GENES = ["MZB1", "PECAM1", "GNLY"]

for sample in SAMPLES:
    spots = combined_adatas[combined_adatas.obs["dataset_name"] == sample]
    sc.pl.spatial(spots, color=GENES, library_id=spots.obs["library_id"].iloc[0], wspace=0.1)